# Deep Learning Training: Breathing Pattern Classification

This notebook provides a **comprehensive, reproducible workflow** for training
deep learning models on the KMC breathing-curve dataset.

### Pipeline Overview

| Stage | Details |
|-------|---------|
| **Input channels** | Volume, dV/dt (breathing rate), d²V/dt², balloon status, amplitude envelope |
| **Windowing** | 2 s windows @ 50 Hz (100 timesteps), 50 % overlap |
| **Normalization** | Per-window Z-score (zero-mean, unit-variance) per channel |
| **Architectures** | LSTM, CNN1D, CNN-LSTM, BiLSTM, GRU, Attention-LSTM, ResNet1D |
| **Training** | Class weighting, EarlyStopping, ReduceLROnPlateau, patient-level split |
| **Evaluation** | Accuracy, Balanced Accuracy, F1, Precision, Recall, Sensitivity, Specificity, MCC, ROC-AUC, PR-AUC |
| **Interpretability** | Grad-CAM / input-gradient saliency |

Modular DL pipeline:
- `src.load_data` — unified data loader
- `src.dl_features` — multi-channel window extraction, signal stats, spectral features
- `src.dl_models` — seven Keras model definitions

> **Alignment with project notebooks:** Derived features (breathing rate = dV/dt,
> smoothed volume = envelope, volume change = diff) follow the feature-engineering
> patterns used across student analysis notebooks.

**Run from the project root or ensure the project root is on `sys.path`.**

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 1. Load Dataset

Uses the **same dataset** as the Jupyter notebooks in `dataset/` (e.g. Abdul Rehaman, Benny Martis, Afsana Sadhick): one folder per patient, `.dat`/`.txt`/`.csv` files with columns Session Time, Volume (liters), Balloon Valve Status, etc. Loaded via `load_all_patients(cfg.DATASET_DIR)` — same layout and file format the notebooks expect.

In [ ]:
import config as cfg
from src.load_data import load_all_patients

df, _ = load_all_patients(cfg.DATASET_DIR, include_session_ini=False)
print(f"Loaded {len(df):,} rows from {df['patient_id'].nunique()} patients")
print(f"Files: {df['file_id'].nunique()}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Build Multi-Channel DL Windows

Extract fixed-length **multi-channel** time-series windows with 50 % overlap.

| Channel | Description | Notebook Alignment |
|---------|-------------|--------------------|
| `Volume (liters)` | Raw breathing signal | All notebooks |
| `vol_derivative` (dV/dt) | Breathing rate / rate of change | Abdul Rehaman — `Volume.diff() / SessionTime.diff()` |
| `vol_derivative2` (d²V/dt²) | Acceleration / rhythm changes | Benny Martis — `Volume Change` via `.diff()` |
| `balloon_numeric` | Hardware signal (1–5) | Afsana Sadhick — anomaly detection on valve status |
| `vol_envelope` | Rolling amplitude envelope (std) | Abdul Rehaman — `Smoothed Volume` via `.rolling().mean()` |

Per-window **Z-score normalization** (zero-mean, unit-variance) is applied per channel.
Signal statistics and spectral features are also computed for analysis.

In [ ]:
from src.dl_features import build_dl_windows, dl_patient_split, MULTI_CHANNELS

CHANNELS = cfg.DL_MULTI_CHANNELS  # 5 channels: Volume, dV/dt, d²V/dt², balloon, envelope
TASK = "breath_hold"  # or "gating_ok"
OVERLAP = cfg.DL_WINDOW_OVERLAP   # 0.5 = 50 % overlap
AUGMENT = cfg.DL_AUGMENT          # jitter augmentation

X, y_bh, y_gk, patient_ids, meta_df = build_dl_windows(
    df,
    channels=CHANNELS,
    window_sec=cfg.WINDOW_SEC,
    sample_rate_hz=cfg.SAMPLE_RATE_HZ,
    min_rows=cfg.MIN_WINDOW_ROWS,
    normalize=True,
    overlap=OVERLAP,
    augment=AUGMENT,
    compute_extras=True,
)

y = y_bh if TASK == "breath_hold" else y_gk

print(f"Windows: {X.shape[0]:,}")
print(f"Window shape: {X.shape[1:]}  (timesteps × channels)")
print(f"Channels: {CHANNELS}")
print(f"Overlap: {OVERLAP*100:.0f}%  |  Augment: {AUGMENT}")
print(f"Label distribution: 0 (free-breathing) = {int((y==0).sum()):,},  "
      f"1 (breath-hold) = {int((y==1).sum()):,}")
print(f"Imbalance ratio: 1:{max(1, int((y==0).sum()) // max(1, int((y==1).sum())))}")
print(f"Unique patients: {len(np.unique(patient_ids))}")

if meta_df is not None and len(meta_df) > 0:
    sig_cols = [c for c in meta_df.columns if c.startswith("sig_") or c.startswith("spec_")]
    if sig_cols:
        print(f"\nPer-window extras computed: {len(sig_cols)} features")
        print(meta_df[sig_cols].describe().round(4))

In [ ]:
# --- Visualize derived channels (matches dataset notebooks: Abdul Rehaman, Benny Martis) ---
from src.dl_features import _add_derived_channels

sample_pid = df["patient_id"].unique()[0]
sample_fid = df[df["patient_id"] == sample_pid]["file_id"].unique()[0]
sample_df = df[(df["patient_id"] == sample_pid) & (df["file_id"] == sample_fid)].copy()
sample_df = sample_df.sort_values("Session Time").head(500)
sample_df = _add_derived_channels(sample_df)

# Include notebook-aligned channels with same names as in dataset notebooks
channel_info = [
    ("Volume (liters)", "Volume (liters) — raw", "steelblue"),
    ("breathing_rate", "Breathing Rate (dV/dT) — Abdul Rehaman formula", "darkorange"),
    ("vol_smoothed", "Smoothed Volume (rolling 5) — Abdul Rehaman / Benny Volume MA", "green"),
    ("vol_derivative", "dV per sample", "gray"),
    ("vol_derivative2", "Acceleration / Volume Change (Benny)", "teal"),
    ("balloon_numeric", "Balloon Valve Status (Afsana Sadhick)", "red"),
    ("vol_envelope", "Amplitude Envelope (rolling σ)", "purple"),
]

fig, axes = plt.subplots(len(channel_info), 1, figsize=(14, 2.2 * len(channel_info)), sharex=True)
fig.suptitle(f"Derived Channels — Patient: {sample_pid}, File: {sample_fid} (dataset notebook–aligned)", fontsize=14, y=1.01)

t = sample_df["Session Time"].values
for ax, (col, label, color) in zip(axes, channel_info):
    if col not in sample_df.columns:
        continue
    vals = pd.to_numeric(sample_df[col], errors="coerce").fillna(0).values
    ax.plot(t, vals, color=color, linewidth=0.8)
    ax.set_ylabel(label, fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Session Time (s)")
plt.tight_layout()
plt.show()
print("Dataset notebook alignment:")
print("  • Abdul Rehaman Untitled1: Breathing Rate = Volume.diff()/SessionTime.diff(), Smoothed Volume = rolling(5).mean()")
print("  • Benny Martis Tumor Movement: Volume Change = .diff(), Volume MA = rolling(5).mean()")
print("  • Afsana Sadhick Untitled1: Balloon Valve Status, time-interval anomalies (mean+2*std)")

### Notebook-style plot (same as Abdul Rehaman feature-engineering output)

The plot below shows **Breathing Rate** and **Smoothed Volume** vs Session Time — matching the feature tables and signal views in `dataset/Abdul Rehaman/Untitled1.ipynb`.

In [ ]:
# Same visual as in dataset/Abdul Rehaman/Untitled1: Breathing Rate and Smoothed Volume
fig2, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
t = sample_df["Session Time"].values
ax1.plot(t, sample_df["breathing_rate"].values, color="darkorange", linewidth=0.9)
ax1.set_ylabel("Breathing Rate (dV/dT)")
ax1.set_title("Breathing Rate — same formula as Abdul Rehaman notebook")
ax1.grid(True, alpha=0.3)
ax2.plot(t, sample_df["vol_smoothed"].values, color="green", linewidth=0.9)
ax2.set_ylabel("Smoothed Volume (rolling 5)")
ax2.set_xlabel("Session Time (s)")
ax2.set_title("Smoothed Volume — same as notebook 'Smoothed Volume' / Benny 'Volume MA'")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Patient-Based Train/Val/Test Split

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = dl_patient_split(
    X, y, patient_ids,
    train_ratio=cfg.TRAIN_RATIO,
    val_ratio=cfg.VAL_RATIO,
    random_state=cfg.RANDOM_STATE,
)

print(f"Train: {len(X_train):,} windows")
print(f"Val:   {len(X_val):,} windows")
print(f"Test:  {len(X_test):,} windows")

## 4. Visualize Sample Windows

In [ ]:
n_channels = X_train.shape[2]
ch_names = CHANNELS[:n_channels]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle(f"Sample Windows ({n_channels} channels, normalized)", fontsize=14)

colors = plt.cm.Set2(np.linspace(0, 1, n_channels))

for row, (label, label_name) in enumerate([(0, "Free-breathing"), (1, "Breath-hold")]):
    idxs = np.where(y_train == label)[0]
    if len(idxs) == 0:
        continue
    samples = np.random.choice(idxs, min(4, len(idxs)), replace=False)
    for col, idx in enumerate(samples):
        for ch_i in range(n_channels):
            axes[row, col].plot(X_train[idx, :, ch_i], color=colors[ch_i],
                                linewidth=0.8, alpha=0.8, label=ch_names[ch_i] if col == 0 else "")
        axes[row, col].set_title(f"{label_name} (idx={idx})", fontsize=9)
        axes[row, col].set_xlabel("Timestep")

axes[0, 0].set_ylabel("Normalized Value")
axes[1, 0].set_ylabel("Normalized Value")
axes[0, 0].legend(fontsize=6, loc="upper right")
plt.tight_layout()
plt.show()

## 5. Build and Train Models

We train **seven architectures** with automatic class weighting to handle label imbalance:

| # | Architecture | Type | Key Feature |
|---|-------------|------|-------------|
| 1 | LSTM | Recurrent | Temporal dependencies |
| 2 | CNN1D | Convolutional | Local pattern extraction |
| 3 | CNN-LSTM | Hybrid | CNN features → LSTM context |
| 4 | BiLSTM | Recurrent | Forward + backward context |
| 5 | GRU | Recurrent | Lighter recurrent (fewer params) |
| 6 | Attention-LSTM | Attention | Temporal self-attention weighting |
| 7 | ResNet1D | Residual | Deep feature extraction with skip connections |

In [ ]:
from src.dl_models import get_model, MODEL_BUILDERS

input_shape = (X_train.shape[1], X_train.shape[2])
print(f"Input shape: {input_shape}")
print(f"Available models: {list(MODEL_BUILDERS.keys())}")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights to handle imbalanced labels
class_weights = None
if cfg.DL_CLASS_WEIGHTS:
    cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
    class_weights = {0: cw[0], 1: cw[1]}
    print(f"Class weights: {class_weights}")

histories = {}
models = {}

for name in MODEL_BUILDERS:
    print(f"\n{'='*60}")
    print(f"Training {name}")
    print(f"{'='*60}")
    
    model = get_model(
        name, input_shape,
        learning_rate=cfg.DL_LEARNING_RATE,
        dropout=cfg.DL_DROPOUT,
    )
    model.summary()
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=cfg.DL_EARLY_STOPPING_PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),
    ]
    
    val_data = (X_val, y_val) if len(X_val) > 0 else None
    
    history = model.fit(
        X_train, y_train,
        validation_data=val_data,
        epochs=cfg.DL_EPOCHS,
        batch_size=cfg.DL_BATCH_SIZE,
        callbacks=callbacks,
        class_weight=class_weights,
        verbose=1,
    )
    
    histories[name] = history.history
    models[name] = model

print(f"\nAll {len(models)} models trained!")

## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(len(histories), 2, figsize=(14, 4 * len(histories)))
if len(histories) == 1:
    axes = axes.reshape(1, -1)

for i, (name, hist) in enumerate(histories.items()):
    epochs_range = range(1, len(hist["loss"]) + 1)
    
    axes[i, 0].plot(epochs_range, hist["loss"], label="Train")
    if "val_loss" in hist:
        axes[i, 0].plot(epochs_range, hist["val_loss"], label="Val")
    axes[i, 0].set_title(f"{name} — Loss")
    axes[i, 0].set_xlabel("Epoch")
    axes[i, 0].set_ylabel("Loss")
    axes[i, 0].legend()
    
    axes[i, 1].plot(epochs_range, hist["accuracy"], label="Train")
    if "val_accuracy" in hist:
        axes[i, 1].plot(epochs_range, hist["val_accuracy"], label="Val")
    axes[i, 1].set_title(f"{name} — Accuracy")
    axes[i, 1].set_xlabel("Epoch")
    axes[i, 1].set_ylabel("Accuracy")
    axes[i, 1].legend()

plt.tight_layout()
plt.show()

## 7. Evaluate on Test Set

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve,
    matthews_corrcoef, average_precision_score, precision_recall_curve,
)

results = {}

for name, model in models.items():
    y_prob = model.predict(X_test, batch_size=cfg.DL_BATCH_SIZE).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    try:
        auc = roc_auc_score(y_test, y_prob)
    except ValueError:
        auc = 0.0
    try:
        pr_auc = average_precision_score(y_test, y_prob)
    except ValueError:
        pr_auc = 0.0
    
    results[name] = {
        "accuracy": acc, "balanced_accuracy": bal_acc,
        "f1": f1, "precision": prec, "recall": rec,
        "sensitivity": sensitivity, "specificity": specificity,
        "mcc": mcc, "roc_auc": auc, "pr_auc": pr_auc,
        "y_prob": y_prob, "y_pred": y_pred,
    }
    
    print(f"\n{'='*50}")
    print(f" {name}")
    print(f"{'='*50}")
    print(f"  Accuracy:          {acc:.4f}")
    print(f"  Balanced Accuracy: {bal_acc:.4f}")
    print(f"  F1 Score:          {f1:.4f}")
    print(f"  Precision:         {prec:.4f}")
    print(f"  Recall:            {rec:.4f}")
    print(f"  Sensitivity (TPR): {sensitivity:.4f}")
    print(f"  Specificity (TNR): {specificity:.4f}")
    print(f"  MCC:               {mcc:.4f}")
    print(f"  ROC-AUC:           {auc:.4f}")
    print(f"  PR-AUC:            {pr_auc:.4f}")
    print()
    print(classification_report(y_test, y_pred, target_names=["Free-breathing", "Breath-hold"]))

## 8. ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, r in results.items():
    try:
        fpr, tpr, _ = roc_curve(y_test, r["y_prob"])
        plt.plot(fpr, tpr, label=f"{name} (AUC={r['roc_auc']:.4f})")
    except ValueError:
        pass

plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves — DL Models")
plt.legend()
plt.tight_layout()
plt.show()

## 8b. Precision-Recall Curves

PR curves are especially informative when classes are imbalanced (many more free-breathing than breath-hold windows).

In [ ]:
plt.figure(figsize=(8, 6))
for name, r in results.items():
    try:
        prec_vals, rec_vals, _ = precision_recall_curve(y_test, r["y_prob"])
        plt.plot(rec_vals, prec_vals, label=f"{name} (PR-AUC={r['pr_auc']:.4f})")
    except ValueError:
        pass

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves — DL Models")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Confusion Matrices

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 5))
if len(results) == 1:
    axes = [axes]

labels_display = ["Free-breathing", "Breath-hold"] if TASK == "breath_hold" else ["Gating Not OK", "Gating OK"]

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r["y_pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels_display, yticklabels=labels_display, ax=ax)
    ax.set_title(f"{name}\nBal.Acc={r['balanced_accuracy']:.4f}")
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")

plt.tight_layout()
plt.show()

## 10. Save Models

Save trained models and metrics so the Streamlit frontend can use them.

In [ ]:
import json

out_dir = cfg.MODELS_DIR
out_dir.mkdir(parents=True, exist_ok=True)

all_metrics = {}

for name, model in models.items():
    prefix = f"dl_{name.lower()}_{TASK}"
    
    # Save model
    model_path = out_dir / f"{prefix}_model.keras"
    model.save(str(model_path))
    print(f"Saved {model_path}")
    
    # Save history
    hist_data = {k: [float(v) for v in vals] for k, vals in histories[name].items()}
    with (out_dir / f"{prefix}_history.json").open("w") as f:
        json.dump(hist_data, f, indent=2)
    
    # Save metrics
    r = results[name]
    m = {
        "model_name": name,
        "task": TASK,
        "accuracy": float(r["accuracy"]),
        "balanced_accuracy": float(r["balanced_accuracy"]),
        "f1": float(r["f1"]),
        "precision": float(r["precision"]),
        "recall": float(r["recall"]),
        "sensitivity": float(r["sensitivity"]),
        "specificity": float(r["specificity"]),
        "mcc": float(r["mcc"]),
        "roc_auc": float(r["roc_auc"]),
        "pr_auc": float(r["pr_auc"]),
        "confusion_matrix": confusion_matrix(y_test, r["y_pred"]).tolist(),
        "epochs_trained": len(histories[name].get("loss", [])),
        "input_shape": list(input_shape),
        "channels": CHANNELS,
        "overlap": OVERLAP,
    }
    all_metrics[name] = m
    with (out_dir / f"{prefix}_metrics.json").open("w") as f:
        json.dump(m, f, indent=2)
    
    # Save test predictions
    preds = {
        "task": TASK,
        "model_name": name,
        "n_test": int(len(y_test)),
        "y_true": y_test.astype(int).tolist(),
        "y_pred": r["y_pred"].tolist(),
        "y_pred_prob": r["y_prob"].tolist(),
    }
    with (out_dir / f"{prefix}_test_predictions.json").open("w") as f:
        json.dump(preds, f, indent=2)

# Save summary
best_name = max(all_metrics, key=lambda k: all_metrics[k]["balanced_accuracy"])
summary = {
    "task": TASK,
    "models": all_metrics,
    "best_model": best_name,
    "best_balanced_accuracy": all_metrics[best_name]["balanced_accuracy"],
}
with (out_dir / f"dl_summary_{TASK}.json").open("w") as f:
    json.dump(summary, f, indent=2)

print(f"\nBest model: {best_name} (bal_acc={summary['best_balanced_accuracy']:.4f})")
print(f"All artifacts saved to {out_dir}")

## 11. Summary Comparison Table

In [ ]:
comparison = []
for name, r in results.items():
    comparison.append({
        "Model": name,
        "Accuracy": f"{r['accuracy']:.4f}",
        "Bal. Acc.": f"{r['balanced_accuracy']:.4f}",
        "F1": f"{r['f1']:.4f}",
        "Precision": f"{r['precision']:.4f}",
        "Recall": f"{r['recall']:.4f}",
        "Sensitivity": f"{r['sensitivity']:.4f}",
        "Specificity": f"{r['specificity']:.4f}",
        "MCC": f"{r['mcc']:.4f}",
        "ROC-AUC": f"{r['roc_auc']:.4f}",
        "PR-AUC": f"{r['pr_auc']:.4f}",
    })

comp_df = pd.DataFrame(comparison)
comp_df.style.highlight_max(subset=comp_df.columns[1:], color="lightgreen")

## 12. Grad-CAM: Model Interpretability

Gradient-weighted Class Activation Mapping (Grad-CAM) highlights **which time steps
the model focuses on** when making predictions. For recurrent/convolutional layers,
we compute gradients of the target class w.r.t. the last Conv/LSTM/GRU layer output.

If the intermediate-layer approach fails, we fall back to **input-gradient saliency**
(gradient w.r.t. the raw input window).

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from frontend.utils.inference import compute_gradcam

best_model = models[best_name]
n_show = min(4, len(X_test))
sample_idxs = np.random.choice(len(X_test), n_show, replace=False)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show))
if n_show == 1:
    axes = [axes]

for ax, idx in zip(axes, sample_idxs):
    window = X_test[idx]
    true_label = int(y_test[idx])
    pred_class = int(results[best_name]["y_pred"][idx])
    pred_prob = float(results[best_name]["y_prob"][idx])

    importance = compute_gradcam(best_model, window, target_class=pred_class)
    timesteps = np.arange(window.shape[0])

    ax.plot(timesteps, window[:, 0], color="steelblue", linewidth=0.9, label="Volume")
    ax.fill_between(timesteps, window[:, 0].min(), window[:, 0].max(),
                     alpha=importance * 0.5, color="red", label="Grad-CAM importance")

    label_name = {0: "Free-breathing", 1: "Breath-hold"}
    ax.set_title(f"True: {label_name[true_label]} | Pred: {label_name[pred_class]} "
                 f"(p={pred_prob:.3f}) — {best_name}", fontsize=10)
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Normalized Volume")
    if idx == sample_idxs[0]:
        ax.legend(fontsize=8)

plt.suptitle(f"Grad-CAM Visualization — {best_name} (best model)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print(f"Red shading = regions the model attends to most when predicting the output class.")

## 13. Notes & Notebook Alignment

### Dataset and loading (same as notebooks)

- **Data source**: `cfg.DATASET_DIR` (project `dataset/` folder), one subfolder per patient, `.dat`/`.txt`/`.csv` inside.
- **Loader**: `load_all_patients(cfg.DATASET_DIR)` — same structure as in **Abdul Rehaman** and **Benny Martis** notebooks (`.dat` with header until `HeaderEnd`, semicolon-separated columns: Session Time, Volume (liters), Balloon Valve Status, etc.).

### Feature engineering (aligned with dataset notebook code)

| Notebook | Code / formula in notebook | Our implementation |
|----------|----------------------------|---------------------|
| **Abdul Rehaman Untitled1** | `Breathing Rate = df["Volume (liters)"].diff() / df["Session Time"].diff()` | `breathing_rate` channel (same formula) |
| **Abdul Rehaman Untitled1** | `Smoothed Volume = df["Volume (liters)"].rolling(window=5, min_periods=1).mean()` | `vol_smoothed` channel (same) |
| **Abdul Rehaman** | `create_sequences(seq_length=10)`, LSTM(50, 50), Dense(2) regression | We use fixed 2 s windows (100 steps), LSTM/CNN etc. for **classification** |
| **Benny Martis Tumor Movement** | `Volume Change = .diff().fillna(0)`, `Volume MA = .rolling(5).mean()` | `vol_derivative`, `vol_smoothed` |
| **Afsana Sadhick Untitled1** | Balloon Valve Status vs time; anomaly = `mean(time_intervals) + 2*std` | `balloon_numeric` channel; anomaly logic can be applied in analysis |

### Notebook figures we align with

- **Abdul Rehaman**: Feature table (Breathing Rate, Smoothed Volume, Breath Phase) — our “Derived Channels” plot reproduces the same signals. Their “LSTM Training Loss” and “Predictions vs Actual” are regression-specific; we have Loss/Accuracy and ROC/PR/confusion matrix for classification.
- **Afsana Sadhick**: “Breathing Cycle” (Balloon vs Session Time), “Time Intervals” with anomaly threshold — our derived-channel plot includes Balloon Valve Status; anomaly threshold can be computed from `np.diff(Session Time)` in analysis.
- **Benny Martis**: Volume, Volume Change, Volume MA — we plot `vol_smoothed` and `vol_derivative`/`vol_derivative2` in the same spirit.

### Key design decisions

- **Classification vs regression**: Dataset notebooks use **regression** (predict next volume or breathing rate). Our pipeline uses **binary classification** (breath-hold vs free-breathing) for the clinical gating task; we reuse the same **features/channels** (Breathing Rate, Smoothed Volume, etc.) for consistency.
- **Normalization**: Notebooks use **MinMaxScaler** on full series. We use **per-window Z-score** so different patients/sessions don’t distort each other; optional MinMax can be added for experiments.
- **Channels**: Default `DL_MULTI_CHANNELS` uses Volume, vol_derivative, vol_derivative2, balloon_numeric, vol_envelope. For exact notebook alignment use `NOTEBOOK_ALIGNED_CHANNELS` in `src.dl_features` (Volume, breathing_rate, vol_smoothed, vol_derivative2, balloon_numeric).
- **Class weighting**: Used to handle imbalance between breath-hold and free-breathing windows.